## Extract blade average data from HDF4 files

### Write it down to info_ file in the coresponding directory

In [10]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path().resolve().parent))

from xbpm_bumps.core.config import Config
from xbpm_bumps.core.data_structure import BladeAvgData
# from xbpm_bumps.core.analysis_info import _blade_data as bldt
import numpy as np
import xarray as xr

In [ ]:
def blade_data(bladedata: "BladeAvgData") -> str:
    """Format the blade map data.

    Args:
        bladedata : "BladeAvgData", the blade map data to format

    Returns:
        A string representing the formatted blade map data.
    """
    lines = []
    # Header and parameter dictionary.
    lines.append("### Blade Averaged Data\n\n")
    for k, val in bladedata.prm.items():
        lines.append(f"# {k:15} : {val}\n")
    lines.append("\n")

    cols = bladedata.blades.table_columns
    names = [n for n, _ in cols]
    arrays = [
        np.full(bladedata.blades.to.size, np.nan)
        if a is None else np.ravel(a)
        for _, a in cols
    ]

    lines.append("  ".join(f"{n:>14}" for n in names) + "\n")
    for row in zip(*arrays):
        lines.append("  ".join(f"{_f(v):>14}" for v in row) + "\n")

    return ''.join(lines)


def _f(
        value  : float,
        digits : int = 2
        ) -> str:
    """Format a float value with a given number of significant digits.

    Args:
        value: The float value to format.
        digits: The number of significant digits.
    
    Returns:
        The formatted float as a string.
    """
    try:
        return f"{float(value):7.{digits}f}"
    except (TypeError, ValueError):
        return str(value)


# def _err(
#         err    : float,
#         digits : int = 3,
#         decs   : int = 1,
#         ) -> str:
#     """Format an error value.

#     Args:
#         err    : The associated error.
#         digits : The number of significant digits.
#         decs   : The number of decimal places for the error.
#     Returns:
#         A string representation of the error.
#     """
#     return f"({float(err):{digits}.{decs}g})"


In [161]:
import h5py
from pathlib import Path

wdir = "/home/arnaldo.filho/XBPM/Measurements/Analysis/revision_2025_2026/"
beamline = "MGN1"

target_dir = Path(wdir + beamline)
pfiles = [
    p.name
    for p in target_dir.iterdir()
    if (
        p.is_file() and
        p.name.startswith("xbpm_analysis_") and
        p.name.endswith(".h5")
        )
    ]


for pfile in pfiles:
    print(f"\nReading file: {pfile}")

    blade_data_lines = ""
    with h5py.File(f"{target_dir}/{pfile}", "r") as f:
        rawdata = f[beamline]['Raw data']
        blavg = BladeAvgData.from_hdf5(rawdata['Blade averages'])
        blavg.prm['Beamline'] = beamline
        blade_data_lines = blade_data(blavg)

        # print(blade_data_lines)

    import re
    pf = re.search(r"_[0-9]+.*", pfile).group()
    pf = re.sub(r'\.h5$', '', pf)
    filename = f"{wdir}{beamline}/{beamline}{pf}/info_Blade_Data.txt"
    try:
        # Uncomment to save data.
        # with open(filename, "w") as f:
        #     f.write(blade_data_lines)
        print(f"Output file: {filename}")
    except Exception as e:
        print(f"\n### Failed to write file {filename}:\n\t### {e}\n")
        print(f"\t >>> Pattern = {pf} \t ({pfile.strip('.h5')})\n")



Reading file: xbpm_analysis_MGN1_2025-07-29_11-14-32.h5
Output file: /home/arnaldo.filho/XBPM/Measurements/Analysis/revision_2025_2026/MGN1/MGN1_2025-07-29_11-14-32/info_Blade_Data.txt

Reading file: xbpm_analysis_MGN1_2026-09-28_20-47-20_pass03.h5
Output file: /home/arnaldo.filho/XBPM/Measurements/Analysis/revision_2025_2026/MGN1/MGN1_2026-09-28_20-47-20_pass03/info_Blade_Data.txt

Reading file: xbpm_analysis_MGN1_2026-06-08_17-43-56_pass03.h5
Output file: /home/arnaldo.filho/XBPM/Measurements/Analysis/revision_2025_2026/MGN1/MGN1_2026-06-08_17-43-56_pass03/info_Blade_Data.txt

Reading file: xbpm_analysis_MGN1_2026-09-28_21-05-00_pass04.h5
Output file: /home/arnaldo.filho/XBPM/Measurements/Analysis/revision_2025_2026/MGN1/MGN1_2026-09-28_21-05-00_pass04/info_Blade_Data.txt

Reading file: xbpm_analysis_MGN1_2026-09-28_16-53-13_pass01.h5
Output file: /home/arnaldo.filho/XBPM/Measurements/Analysis/revision_2025_2026/MGN1/MGN1_2026-09-28_16-53-13_pass01/info_Blade_Data.txt

Reading file: